# AI Agents: Fundamentals
## with LangChain & LangGraph (using Ollama llama3.2)

A hands-on teaching notebook. We start from a plain LLM, see where it breaks, and build up to a
**tool-using agent with memory and human approval**, first using LangChain's prebuilt agent and
then from scratch with LangGraph.

**What you will learn**
1. **What an agent is**: the reasoning loop (think → act → observe)
2. **How agents differ from plain LLMs**, shown with a live demo
3. **Agent components**: LLM, tools, memory (short- and long-term), planning, state
4. **A simple agent in LangChain** with `create_agent`
5. **An agent from scratch in LangGraph**: state, nodes, edges, conditional routing, streaming
6. **Memory in LangGraph**: checkpointers (per thread) and stores (across threads)
7. **Production concerns**: step limits, tool errors, human-in-the-loop approval

Everything runs **locally** on Ollama. No API keys are needed.

## Setup

### 1. Install the Python packages

In [ ]:
%pip install -q -U langchain langchain-ollama langgraph grandalf

### 2. Make sure Ollama is running

In a terminal (outside this notebook):
```bash
ollama pull llama3.2
ollama serve   # usually already running as a background service
```

`llama3.2` supports **tool calling**, which is what lets it act as an agent.

### 3. Initialize the LLM

In [1]:
from langchain_ollama import ChatOllama

# temperature=0 gives more deterministic tool choices (useful when teaching)
llm = ChatOllama(model="llama3.2", temperature=0)

print(llm.invoke("Say hello in one word.").content)

Hello.


---
## 1. What is an Agent?

> An **agent** is a system where an **LLM decides the control flow**: it chooses which actions
> (tools) to take, looks at the results, and repeats until the goal is met.

A plain LLM call is *one step*: text in, text out. An agent wraps the LLM in a **loop**:

```
            ┌──────────────────────────────────────────┐
            │                                          │
  user ──▶  │   ┌─────────┐  tool call   ┌─────────┐   │
  goal      │   │   LLM   │ ───────────▶ │  TOOLS  │   │
            │   │ (think) │              │  (act)  │   │
            │   └─────────┘ ◀─────────── └─────────┘   │
            │        │       tool result               │
            │        │       (observe)                 │
            └────────┼─────────────────────────────────┘
                     │ no more tool calls
                     ▼
               final answer
```

This pattern is called **ReAct** (*Reason + Act*, [Yao et al., 2022](https://arxiv.org/abs/2210.03629)):
1. **Reason**: the LLM thinks about what it needs next
2. **Act**: it emits a structured *tool call* (e.g. `calculator(expression="23.7*891")`)
3. **Observe**: our code runs the tool and feeds the result back to the LLM
4. **Repeat** until the LLM answers without calling a tool

### Levels of autonomy

"Agentic" is a spectrum. The more the LLM decides, the more agentic the system is:

| Level | Who decides the steps? | Example |
|---|---|---|
| Single LLM call | Developer (one fixed step) | "Summarize this text" |
| Chain | Developer (fixed sequence) | Retrieve → prompt → LLM (basic RAG) |
| Router | LLM picks **one** branch | Classify query → send to billing or tech support |
| **Tool-calling agent** | LLM picks tools **in a loop** | This notebook |
| Multi-agent | LLMs delegate to other agents | Supervisor + researcher + writer |

> Docs: [LangGraph: Agentic concepts](https://langchain-ai.github.io/langgraph/concepts/agentic_concepts/)

---
## 2. How Agents Differ from Plain LLMs

| Aspect | Plain LLM | Agent |
|---|---|---|
| **Control flow** | One call, fixed by the developer | Loop; the LLM decides the next step |
| **Actions** | Can only generate text | Can call tools (APIs, DBs, code, search) |
| **Knowledge** | Frozen at training cutoff | Can fetch live or private data through tools |
| **Math / precision** | Predicts digits (often wrong) | Delegates to a calculator |
| **State / memory** | Stateless; you resend history yourself | Keeps state across steps and sessions |
| **Termination** | After one response | When the goal is met (or a step limit is hit) |
| **Cost / latency** | 1 LLM call | N LLM calls + tool calls |
| **Risk** | Wrong text | Wrong **actions** (needs guardrails and approval) |

### Demo: the plain LLM on its own

We ask something that needs **exact arithmetic** and **today's date**. Neither is available
to a model that only predicts the next token.

In [3]:
QUESTION = "What is 23.7 * 891 + 1024, and what is today's date?"

print(f"Correct math answer: {23.7 * 891 + 1024}\n")
print("Plain LLM says:")
print(llm.invoke(QUESTION).content)

Correct math answer: 22140.7

Plain LLM says:
I'd be happy to help you with the calculation.

23.7 * 891 = 21155.7

Adding 1024 to that result:

21155.7 + 1024 = 22179.7

As for today's date, I'm a large language model, I don't have have access to real-time information or a physical clock. I can suggest ways for you to find out the current date, such as:

* Checking your device's clock or calendar app
* Searching for "current date" on a search engine
* Asking a voice assistant like Siri, Google Assistant, or Alexa

Please let me know if you have any other questions or if there's anything else I can help you with!


The model will usually get the arithmetic slightly wrong, make up a date, or admit it doesn't know.
Keep this question in mind: we'll ask the **agent** the same thing in Section 4.

---
## 3. Agent Components

```
 ┌───────────────────────────── AGENT ─────────────────────────────┐
 │                                                                 │
 │   ┌──────────────┐   instructions / ReAct prompt  (PLANNING)    │
 │   │  LLM "brain" │◀──────────────────────────────                │
 │   └──────┬───────┘                                              │
 │          │ reads / writes                                       │
 │   ┌──────▼───────┐        ┌───────────────────────────────┐     │
 │   │    STATE     │        │            MEMORY             │     │
 │   │ (messages,   │◀──────▶│ short-term: thread history    │     │
 │   │  scratchpad) │        │ long-term : facts across chats│     │
 │   └──────┬───────┘        └───────────────────────────────┘     │
 │          │ tool calls                                           │
 │   ┌──────▼───────┐                                              │
 │   │    TOOLS     │  calculator · date · weather · KB search ... │
 │   └──────────────┘                                              │
 └─────────────────────────────────────────────────────────────────┘
```

| Component | Role | In LangChain / LangGraph |
|---|---|---|
| **LLM** | Reasons, picks tools, writes the answer | `ChatOllama(...).bind_tools(tools)` |
| **Tools** | Let the agent act on the world | `@tool` functions, `ToolNode` |
| **Memory (short-term)** | Conversation history within a thread | Checkpointer (`InMemorySaver`) + `thread_id` |
| **Memory (long-term)** | Facts that persist across threads | Store (`InMemoryStore`) |
| **Planning** | How the model decides what to do | System prompt, ReAct loop |
| **State & control flow** | What data flows where, and when to stop | `StateGraph`, nodes, edges |

We'll build each one in turn.

### 3a. Tools

A tool is just a Python function with a **name, a description, and typed arguments**. The
`@tool` decorator turns the signature and docstring into a JSON schema the LLM can read.

> **The docstring is the prompt.** The LLM decides *whether* and *how* to call a tool purely from
> its name, description, and argument schema, so write them carefully.

In [14]:
import ast
import operator
from datetime import date
from langchain_core.tools import tool

# --- A safe calculator (no eval() on raw user input!) -----------------------
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
    ast.Mod: operator.mod,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")


@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression exactly, e.g. '23.7 * 891 + 1024'.
    Supports + - * / ** % and parentheses. Use this for ANY math."""
    result = _safe_eval(ast.parse(expression, mode="eval").body)
    return str(result)


@tool
def get_current_date() -> str:
    """Return today's date in ISO format (YYYY-MM-DD)."""
    return str(date.today().isoformat())


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Fake data so the notebook runs offline; swap in a real API in practice.
    fake = {"london": "12°C, light rain", "paris": "18°C, sunny", "tokyo": "22°C, cloudy"}
    return fake.get(city.lower(), f"No weather data for {city}.")


KNOWLEDGE_BASE = {
    "refund policy": "Refunds are available within 30 days of purchase with a receipt.",
    "support hours": "Support is open Monday-Friday, 9am-6pm UTC.",
    "shipping": "Standard shipping takes 3-5 business days; express takes 1-2.",
}

@tool
def search_knowledge_base(query: str) -> str:
    """Search the company knowledge base for policies: refunds, support hours, shipping."""
    hits = [v for k, v in KNOWLEDGE_BASE.items() if any(w in query.lower() for w in k.split())]
    print(f"search_knowledge_base({query}) -> {hits}")
    return "\n".join(hits) or "No matching policy found."


tools = [calculator, get_current_date, get_weather, search_knowledge_base]

# Tools are Runnables: you can call them directly
print(calculator.invoke({"expression": "23.7 * 891 + 1024"}))
print(get_weather.invoke({"city": "Paris"}))

22140.7
18°C, sunny


#### What the LLM actually sees

This is the schema sent to the model for each tool:

In [8]:
import json

print(json.dumps(calculator.tool_call_schema.model_json_schema(), indent=2))

{
  "description": "Evaluate an arithmetic expression exactly, e.g. '23.7 * 891 + 1024'.\nSupports + - * / ** % and parentheses. Use this for ANY math.",
  "properties": {
    "expression": {
      "title": "Expression",
      "type": "string"
    }
  },
  "required": [
    "expression"
  ],
  "title": "calculator",
  "type": "object"
}


### 3b. The LLM brain: `bind_tools`

`bind_tools` attaches the tool schemas to every request. The model can then answer in one of two ways:
- **plain text** (`content`), or
- **one or more tool calls** (`tool_calls`): structured requests like `{"name": "calculator", "args": {...}}`

**The LLM never runs the tool itself.** It only *asks* for it. Running the tool is our code's job
(or LangGraph's `ToolNode`, as we'll see).

In [9]:
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("What's the weather in Tokyo?")
print("content   :", repr(response.content))
print("tool_calls:", response.tool_calls)
print(response.content)

content   : ''
tool_calls: [{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'c5a70bd5-c7e5-4089-b732-086aeaccdae2', 'type': 'tool_call'}]



In [15]:
response = llm_with_tools.invoke("What's current date and time?")
print("content   :", repr(response.content))
print("tool_calls:", response.tool_calls)
print(response.content)

content   : ''
tool_calls: [{'name': 'get_current_date', 'args': {}, 'id': '75db8193-0f73-47b2-98a8-43a1ece5ccd0', 'type': 'tool_call'}]



### 3c. The agent loop by hand

Before using any framework, let's write the ReAct loop ourselves in about 15 lines. This is
**all an agent is**. LangGraph turns this loop into an explicit, inspectable, persistent graph.

In [16]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

TOOLS_BY_NAME = {t.name: t for t in tools}

def manual_agent(question: str, max_steps: int = 5) -> str:
    messages = [
        SystemMessage("You are a helpful assistant. Use tools for math, dates, weather and policies."),
        HumanMessage(question),
    ]
    for step in range(max_steps):
        ai_msg = llm_with_tools.invoke(messages)          # 1. REASON
        messages.append(ai_msg)
        if not ai_msg.tool_calls:                         # no tool needed, so we're done
            return ai_msg.content
        for call in ai_msg.tool_calls:                    # 2. ACT
            result = TOOLS_BY_NAME[call["name"]].invoke(call["args"])
            print(f"  step {step}: {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))  # 3. OBSERVE
    return "Stopped: step limit reached."

print(manual_agent("What is 23.7 * 891 + 1024?"))

  step 0: calculator({'expression': '23.7 * 891 + 1024'}) -> 22140.7
The result of the calculation is 22140.7.


### 3d. Memory: the two kinds

| | Short-term memory | Long-term memory |
|---|---|---|
| **Scope** | One conversation (a *thread*) | Across all threads for a user |
| **Holds** | The message history / graph state | Facts, preferences, past learnings |
| **LangGraph primitive** | **Checkpointer** (`InMemorySaver`, `SqliteSaver`, `PostgresSaver`) | **Store** (`InMemoryStore`, `PostgresStore`) |
| **Keyed by** | `thread_id` | a namespace, e.g. `("memories", user_id)` |
| **Analogy** | Working memory | A notebook the agent writes in |

Our `manual_agent` above has **no memory**: every call starts with a fresh `messages` list.
We'll add both kinds in Section 5.

### 3e. Planning and state
- **Planning** in a ReAct agent is implicit: the system prompt plus the model's reasoning at every
  loop step. More advanced agents add an explicit *plan-and-execute* step.
- **State** is the data that flows through the agent (here, the `messages` list). In LangGraph
  you declare it explicitly as a typed schema.

---
## 4. A Simple Agent with LangChain (prebuilt)

LangChain's `create_agent` gives you a production-ready ReAct loop in one call.
Under the hood it **builds a LangGraph graph**, just like the one we'll write by hand next.

> Docs: [LangChain: Agents](https://docs.langchain.com/oss/python/langchain/agents)

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "You are a helpful assistant. "
    "Always use the calculator for arithmetic and get_current_date for today's date. "
    "Answer concisely."
)

simple_agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

result = simple_agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print(result["messages"][-1].content)

This is the same question the plain LLM got wrong in Section 2. Let's look at the **full message
trace** to see the think → act → observe loop:

In [ ]:
def show_trace(messages):
    for m in messages:
        if m.type == "ai" and m.tool_calls:
            for c in m.tool_calls:
                print(f"🤖 AI   → call {c['name']}({c['args']})")
        elif m.type == "ai":
            print(f"🤖 AI   → {m.content}")
        elif m.type == "tool":
            print(f"🔧 TOOL ← {m.name}: {m.content}")
        else:
            print(f"👤 {m.type.upper():4} : {m.content}")

show_trace(result["messages"])

The prebuilt agent is itself a LangGraph graph. Here is its structure:

In [ ]:
print(simple_agent.get_graph().draw_ascii())

---
## 5. Building an Agent from Scratch with LangGraph

`create_agent` is convenient, but real systems need custom steps: guardrails, routing, approvals,
retrieval, multiple agents. **LangGraph** lets you build the loop explicitly as a **graph**.

### LangGraph core concepts

| Concept | What it is |
|---|---|
| **State** | A typed dict shared by all nodes; the agent's working memory |
| **Reducer** | How a node's update is merged into state (`add_messages` **appends** instead of overwriting) |
| **Node** | A Python function `state -> partial state update` |
| **Edge** | A fixed transition `A → B` |
| **Conditional edge** | A function that picks the next node at runtime (this is where the LLM "decides") |
| **`START` / `END`** | Special entry and exit points |
| **Compile** | Turns the builder into a runnable app (and attaches memory) |

We'll build this graph:

```
 START ──▶ agent ──(tool calls?)──yes──▶ tools
             ▲          │                  │
             │          no                 │
             │          ▼                  │
             │         END                 │
             └─────────────────────────────┘
```

> Docs: [LangGraph: Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)

### Step 1: Define the state

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    # add_messages is a *reducer*: when a node returns {"messages": [new_msg]},
    # it's APPENDED to the list (and de-duplicated by id) rather than replacing it.
    messages: Annotated[list[AnyMessage], add_messages]

### Step 2: Define the nodes

- **`agent`**: calls the tool-bound LLM with the current messages
- **`tools`**: LangGraph's prebuilt `ToolNode` runs every tool call in the last AI message and
  returns the `ToolMessage` results

In [ ]:
from langgraph.prebuilt import ToolNode

def agent_node(state: AgentState) -> dict:
    messages = [SystemMessage(SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}          # appended thanks to add_messages

# handle_tool_errors=True: if a tool raises, send the error text back to the LLM
# instead of crashing the graph (more on this in Section 6b)
tool_node = ToolNode(tools, handle_tool_errors=True)

### Step 3: Wire the graph

`tools_condition` is a prebuilt router: it returns `"tools"` if the last message contains tool
calls, otherwise `END`. We write our own version too so you can see there's no magic.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition

# What tools_condition does, written out by hand:
def route_after_agent(state: AgentState) -> str:
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)   # or: route_after_agent
builder.add_edge("tools", "agent")                        # loop back after acting

graph = builder.compile()

### Step 4: Visualize

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))   # needs internet (mermaid.ink)
except Exception:
    print(graph.get_graph().draw_ascii())

### Step 5: Run it

In [ ]:
result = graph.invoke({"messages": [HumanMessage("What's the weather in London and Paris?")]})
show_trace(result["messages"])

### Step 6: Stream the steps

`.stream(..., stream_mode="updates")` yields **each node's output as it happens**, which is great
for UIs and debugging. You can watch the agent move between nodes.

In [ ]:
inputs = {"messages": [HumanMessage("What is our refund policy, and what is 15% of 240?")]}

for update in graph.stream(inputs, stream_mode="updates"):
    for node_name, node_output in update.items():
        print(f"── node: {node_name}")
        show_trace(node_output["messages"])

---
### 5b. Short-term memory: checkpointers

Without memory, a follow-up question has no context:

In [ ]:
graph.invoke({"messages": [HumanMessage("Hi, my name is Asha.")]})
out = graph.invoke({"messages": [HumanMessage("What is my name?")]})
print(out["messages"][-1].content)

A **checkpointer** saves the full graph state after every step, keyed by `thread_id`.
When you invoke again with the same `thread_id`, LangGraph loads the previous state and the new
message is **appended** to it (thanks to the `add_messages` reducer).

`InMemorySaver` is for demos. In production use `SqliteSaver` or `PostgresSaver`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
memory_graph = builder.compile(checkpointer=checkpointer)

thread_1 = {"configurable": {"thread_id": "thread-1"}}

memory_graph.invoke({"messages": [HumanMessage("Hi, my name is Asha.")]}, thread_1)
out = memory_graph.invoke({"messages": [HumanMessage("What is my name?")]}, thread_1)
print("thread-1:", out["messages"][-1].content)

# A different thread_id means a fresh conversation with no shared history
thread_2 = {"configurable": {"thread_id": "thread-2"}}
out = memory_graph.invoke({"messages": [HumanMessage("What is my name?")]}, thread_2)
print("thread-2:", out["messages"][-1].content)

Because every step is checkpointed, you can **inspect the saved state** at any time. The same
mechanism enables time travel, resuming after crashes, and human-in-the-loop.

In [ ]:
snapshot = memory_graph.get_state(thread_1)
print(f"{len(snapshot.values['messages'])} messages saved in thread-1")
print("next node to run:", snapshot.next or "(finished)")

### 5c. Long-term memory: the Store

Checkpointer memory is tied to one thread. To remember facts about a **user across conversations**,
LangGraph provides a **Store**: a key-value database organised by namespaces.

We'll give the agent:
1. A `save_memory` **tool** so the LLM itself decides what's worth remembering
2. An agent node that **loads the user's memories** into the system prompt on every turn

Tools and nodes get access to the store through `runtime` (injected by LangGraph, and hidden
from the LLM's view of the tool schema).

In [ ]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore
import uuid


@dataclass
class UserContext:
    """Per-invocation context (who is calling). Passed via invoke(..., context=...)."""
    user_id: str


@tool
def save_memory(fact: str, runtime: ToolRuntime[UserContext]) -> str:
    """Save an important, long-lived fact about the user (name, preferences, location)
    so it can be recalled in future conversations."""
    user_id = runtime.context.user_id
    runtime.store.put(("memories", user_id), str(uuid.uuid4()), {"fact": fact})
    return f"Saved: {fact}"


lt_tools = tools + [save_memory]
llm_lt = llm.bind_tools(lt_tools)


def agent_with_memory(state: AgentState, runtime: Runtime[UserContext]) -> dict:
    user_id = runtime.context.user_id
    memories = runtime.store.search(("memories", user_id))
    known = "\n".join(f"- {m.value['fact']}" for m in memories) or "(nothing yet)"

    system = (
        f"{SYSTEM_PROMPT}\n"
        "When the user shares a personal fact or preference, call save_memory.\n"
        f"Known facts about this user:\n{known}"
    )
    response = llm_lt.invoke([SystemMessage(system)] + state["messages"])
    return {"messages": [response]}


lt_builder = StateGraph(AgentState, context_schema=UserContext)
lt_builder.add_node("agent", agent_with_memory)
lt_builder.add_node("tools", ToolNode(lt_tools, handle_tool_errors=True))
lt_builder.add_edge(START, "agent")
lt_builder.add_conditional_edges("agent", tools_condition)
lt_builder.add_edge("tools", "agent")

store = InMemoryStore()
lt_graph = lt_builder.compile(checkpointer=InMemorySaver(), store=store)

Now we tell the agent a preference in **thread A**, then ask about it in a brand-new **thread B**.
The thread history is empty in B, but the store still has the fact.

In [ ]:
USER = "user-42"

cfg_a = {"configurable": {"thread_id": "A"}}
out = lt_graph.invoke(
    {"messages": [HumanMessage("Please remember that I prefer temperatures in Fahrenheit and I live in Paris.")]},
    cfg_a, context=UserContext(user_id=USER),
)
show_trace(out["messages"])

print("\n📦 Store contents:")
for item in store.search(("memories", USER)):
    print("  -", item.value["fact"])

In [ ]:
cfg_b = {"configurable": {"thread_id": "B"}}   # new thread, no history
out = lt_graph.invoke(
    {"messages": [HumanMessage("What's the weather where I live?")]},
    cfg_b, context=UserContext(user_id=USER),
)
show_trace(out["messages"])

> **Note on small models:** `llama3.2` (3B) doesn't always call `save_memory` when it should, or
> may ignore the Fahrenheit preference. That's the model's limitation, not the pattern's. Larger
> models are much more reliable. The architecture (store + namespaces + injection into the prompt)
> is the same one production agents use.

---
## 6. Production Concerns

An agent chooses its own actions, so you must bound **how long it runs**, **what happens when
tools fail**, and **which actions need a human's approval**.

### 6a. Step limits: `recursion_limit`

Each node execution counts as one step. If the agent loops too long, LangGraph raises
`GraphRecursionError` instead of burning tokens forever. The default is high, so set it explicitly.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    graph.invoke(
        {"messages": [HumanMessage("Get the weather in London, Paris and Tokyo, then add their temperatures.")]},
        {"recursion_limit": 3},    # deliberately tiny to trigger the limit
    )
except GraphRecursionError as e:
    print("Stopped by recursion limit ✋", type(e).__name__)

### 6b. Tool errors

Tools fail: APIs time out, inputs are malformed, and a model might ask for `10 / 0`. With
`handle_tool_errors=True` (set in Step 2), `ToolNode` **catches the exception** and returns the
error text to the LLM as a `ToolMessage`, so the agent can recover (retry, rephrase, or explain)
instead of crashing.

> Without it, LangGraph 1.x `ToolNode` only catches *argument validation* errors, and any other
> exception stops the whole graph.

To keep the demo deterministic, we fake the LLM's tool call and run just the `tools` node.

In [ ]:
from langchain_core.messages import AIMessage

# Simulate the LLM asking for a bad calculation
bad_call = AIMessage(content="", tool_calls=[
    {"name": "calculator", "args": {"expression": "10 / 0"}, "id": "call-1"}
])
tools_only = StateGraph(AgentState)
tools_only.add_node("tools", tool_node)
tools_only.add_edge(START, "tools")
tools_only.add_edge("tools", END)

out = tools_only.compile().invoke({"messages": [bad_call]})
print(out["messages"][-1].content)   # the LLM would see this and can explain / retry

### 6c. Human-in-the-loop: approve risky actions

Some tools have **side effects** (sending email, making payments, deleting data). LangGraph's
`interrupt()` **pauses the graph**, saves its state through the checkpointer, and waits for a human.
You resume with `Command(resume=...)`.

```
agent ──▶ tools ──▶ send_email ──▶ interrupt() ⏸
                                        │
                        human reviews   │  Command(resume="approve" | "reject")
                                        ▼
                               execute or cancel ──▶ agent ──▶ END
```

This connects directly to the guardrails notebooks: approval is the last line of defence for
irreversible actions.

In [ ]:
from langgraph.types import interrupt, Command


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    decision = interrupt({                       # ⏸ pauses the whole graph here
        "action": "send_email",
        "to": to, "subject": subject, "body": body,
        "question": "Approve sending this email? (approve / reject)",
    })
    if decision != "approve":
        return "Email was NOT sent: rejected by human reviewer."
    return f"Email sent to {to}."   # a real implementation would call an email API here


hitl_tools = tools + [send_email]
llm_hitl = llm.bind_tools(hitl_tools)

HITL_PROMPT = (
    "You are an assistant that can send emails with the send_email tool. "
    "After a tool returns, reply to the user in one plain sentence saying what happened."
)

def hitl_agent(state: AgentState) -> dict:
    return {"messages": [llm_hitl.invoke([SystemMessage(HITL_PROMPT)] + state["messages"])]}

hitl_builder = StateGraph(AgentState)
hitl_builder.add_node("agent", hitl_agent)
hitl_builder.add_node("tools", ToolNode(hitl_tools))
hitl_builder.add_edge(START, "agent")
hitl_builder.add_conditional_edges("agent", tools_condition)
hitl_builder.add_edge("tools", "agent")

hitl_graph = hitl_builder.compile(checkpointer=InMemorySaver())   # interrupts REQUIRE a checkpointer

In [ ]:
cfg = {"configurable": {"thread_id": "hitl-1"}}

out = hitl_graph.invoke(
    {"messages": [HumanMessage("Send an email to bob@example.com with subject 'Hello' and body 'See you at 5pm'.")]},
    cfg,
)

if "__interrupt__" in out:
    pending = out["__interrupt__"][0].value
    print("⏸ Graph paused, waiting for approval:")
    print(json.dumps(pending, indent=2))
else:
    print("The model did not call send_email this time. Re-run the cell.")
    show_trace(out["messages"])

In [ ]:
# The human decides. Try changing this to "reject".
HUMAN_DECISION = "approve"

if hitl_graph.get_state(cfg).next:          # only resume if actually paused
    out = hitl_graph.invoke(Command(resume=HUMAN_DECISION), cfg)
    show_trace(out["messages"])

Now the **reject** path, on a fresh thread. Pause, then resume with `"reject"`:

In [ ]:
cfg_reject = {"configurable": {"thread_id": "hitl-2"}}

out = hitl_graph.invoke(
    {"messages": [HumanMessage("Email alice@example.com with subject 'Invoice' and body 'Please pay now'.")]},
    cfg_reject,
)
if "__interrupt__" in out:
    out = hitl_graph.invoke(Command(resume="reject"), cfg_reject)
show_trace(out["messages"])

> **Key lesson: enforce safety in code, not in the prompt.** Look at the `🔧 TOOL` line: the
> email was **not** sent because the `send_email` function checked the human's decision. A small
> model like `llama3.2` may still *word* its final reply badly, and might even claim the email
> went out. The model's text can be wrong, but the **action** was blocked, and that's what matters.
> The same principle applies to the guardrails notebooks.

> **Prebuilt alternative:** with `create_agent` you get the same behaviour with no custom code by
> adding `HumanInTheLoopMiddleware(interrupt_on={"send_email": True})` to `middleware=[...]`.

---
## 7. Recap

| Concept | Key takeaway | Code |
|---|---|---|
| Agent | An LLM in a loop that chooses its own actions | ReAct: think → act → observe |
| Agent vs LLM | Adds tools, state, memory, multi-step control flow | Section 2 demo |
| Tools | Typed functions; the docstring is the prompt | `@tool`, `bind_tools`, `ToolNode` |
| Prebuilt agent | Fastest way to start | `create_agent(model, tools, system_prompt=...)` |
| LangGraph | Explicit, customisable agent graphs | `StateGraph`, nodes, (conditional) edges |
| State | Typed shared data; reducers merge updates | `Annotated[list, add_messages]` |
| Short-term memory | Per-thread history | `compile(checkpointer=...)` + `thread_id` |
| Long-term memory | Cross-thread facts | `compile(store=...)`, `runtime.store` |
| Safety | Bound steps, handle errors, approve risky actions | `recursion_limit`, `ToolNode`, `interrupt()` |

## Exercises

1. **Add a tool**: write a `convert_currency(amount, from_ccy, to_ccy)` tool with fixed rates and
   ask the agent a question that needs both it and the calculator.
2. **Add a node**: insert an `input_guard` node between `START` and `agent` that blocks
   prompt-injection attempts (reuse the rule-based guard from the guardrails notebook).
3. **Change memory scope**: namespace long-term memories by `(org_id, user_id)` and show that
   two users in the same org don't see each other's facts.
4. **Persistent memory**: swap `InMemorySaver` for `SqliteSaver` (`pip install langgraph-checkpoint-sqlite`)
   and confirm the conversation survives a kernel restart.

### Further reading
- [LangGraph documentation](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangChain agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangGraph memory concepts](https://docs.langchain.com/oss/python/langgraph/memory)
- [ReAct paper](https://arxiv.org/abs/2210.03629) · [Anthropic: Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)